In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# Batch Normalization — Section 8.3.0.1

**Batch Normalization** normalizes activations across examples within a mini-batch, then applies a learnable scale (γ) and shift (β).

**Step 1:** Compute mini-batch statistics
$$\mu_B = \frac{1}{m}\sum_{i=1}^m x_i, \qquad \sigma_B^2 = \frac{1}{m}\sum_{i=1}^m (x_i - \mu_B)^2$$

**Step 2:** Normalize
$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \varepsilon}}$$

**Step 3:** Scale and shift
$$y_i = \gamma \hat{x}_i + \beta$$

γ and β are **learnable parameters** — they allow the network to recover the original distribution if normalization is not beneficial.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def batch_norm(x, gamma, beta, eps):
    mu = x.mean()
    sigma2 = x.var()
    xhat = (x - mu) / np.sqrt(sigma2 + eps)
    y = gamma * xhat + beta
    return xhat, y, mu, sigma2


# Fixed batch sampled from a skewed distribution
def get_batch(m, seed=42):
    rng = np.random.default_rng(seed)
    return rng.normal(loc=3.0, scale=2.0, size=m)


def draw_batch_norm(m=8, gamma=1.0, beta=0.0, log_eps=-5):
    eps = 10 ** log_eps
    x = get_batch(m)
    xhat, y, mu, sigma2 = batch_norm(x, gamma, beta, eps)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # Input
    axes[0].hist(x, bins=8, color='#2563eb', edgecolor='white', alpha=0.8)
    axes[0].axvline(mu, color='#dc2626', lw=2, ls='--', label=f'μ={mu:.2f}')
    axes[0].set_title('Input x', fontsize=11)
    axes[0].set_xlabel('Value'); axes[0].set_ylabel('Count')
    axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

    # Normalized
    mu_hat, sigma2_hat = xhat.mean(), xhat.var()
    axes[1].hist(xhat, bins=8, color='#d97706', edgecolor='white', alpha=0.8)
    axes[1].axvline(mu_hat, color='#dc2626', lw=2, ls='--', label=f'μ≈{mu_hat:.2e}')
    axes[1].set_title('Normalized x̂', fontsize=11)
    axes[1].set_xlabel('Value'); axes[1].set_ylabel('Count')
    axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
    axes[1].text(0.02, 0.97, f'var≈{sigma2_hat:.4f}', transform=axes[1].transAxes,
                 fontsize=9, va='top', color='#475569')

    # Output
    mu_y, sigma2_y = y.mean(), y.var()
    axes[2].hist(y, bins=8, color='#059669', edgecolor='white', alpha=0.8)
    axes[2].axvline(mu_y, color='#dc2626', lw=2, ls='--', label=f'μ={mu_y:.2f}')
    axes[2].set_title(f'Output y = γx̂ + β  (γ={gamma}, β={beta})', fontsize=11)
    axes[2].set_xlabel('Value'); axes[2].set_ylabel('Count')
    axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)
    axes[2].text(0.02, 0.97, f'var={sigma2_y:.3f}', transform=axes[2].transAxes,
                 fontsize=9, va='top', color='#475569')

    plt.tight_layout()
    plt.show()

    print(f'\nBatch size m={m},  γ={gamma},  β={beta},  ε=1e{log_eps}')
    print(f'  Step 1: μ_B={mu:.4f},  σ²_B={sigma2:.4f},  σ_B={np.sqrt(sigma2):.4f}')
    print(f'  Step 2: x̂ mean={mu_hat:.2e},  x̂ var={sigma2_hat:.4f}  (≈0 mean, unit variance)')
    print(f'  Step 3: y mean={mu_y:.4f},  y var={sigma2_y:.4f},  y std={np.sqrt(sigma2_y):.4f}')
    if gamma == 1 and beta == 0:
        print('  → γ=1, β=0: identity — output is pure x̂ (zero mean, unit variance)')
    else:
        print(f'  → The network has learned to rescale/shift: mean≈{mu_y:.2f}, std≈{np.sqrt(sigma2_y):.2f}')


widgets.interact(
    draw_batch_norm,
    m=widgets.IntSlider(min=4, max=16, step=2, value=8, description='Batch size m', continuous_update=False),
    gamma=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='γ (scale)', continuous_update=False),
    beta=widgets.FloatSlider(min=-2.0, max=2.0, step=0.1, value=0.0, description='β (shift)', continuous_update=False),
    log_eps=widgets.IntSlider(min=-8, max=-1, step=1, value=-5, description='log₁₀(ε)', continuous_update=False),
)